# Reproduce paper tables from released results

This notebook independently verifies every number in Tables 1, 2, and 3 of
`paper/main.tex`, plus the derived prose claims in Sections 5.1, 5.4, and 5.6,
**without re-running training or any GPU step**.

**Data sources (read-only):**
- `results/raw/` — primary per-seed JSON for temperature-scaling metrics (27 files).
- `results/raw_v2/` — same runs with isotonic fields, and the independent BERT-large SNLI checkpoint used in Tables 2–3.

It does **not** call `experiments/aggregate.py`; aggregation is reimplemented below as a cross-check. Nothing under `results/` is written.


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

# Resolve repo root whether launched from repo root or experiments/
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO = None
for cand in candidates:
    if (cand / "results" / "raw").exists() and (cand / "paper" / "main.tex").exists():
        REPO = cand
        break
assert REPO is not None, f"Could not find repo root from {cwd}"

RAW = REPO / "results" / "raw"
RAW_V2 = REPO / "results" / "raw_v2"
TEX = REPO / "paper" / "main.tex"
print("REPO:", REPO)


REPO: /home/txdigitalafrica/temperature-scaling-research


## 1. Load every file in `results/raw/`


In [2]:
def load_raw_dir(d: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(d.glob("results_*.json")):
        with open(p) as f:
            r = json.load(f)
        r["_file"] = p.name
        if "dataset" not in r:
            r["dataset"] = "CIFAR-10H" if "resnet" in r["model"] else None
        rows.append(r)
    return pd.DataFrame(rows)

df_raw = load_raw_dir(RAW)
df_v2 = load_raw_dir(RAW_V2)
print(f"raw/: {len(df_raw)} files; raw_v2/: {len(df_v2)} files")
assert len(df_raw) == 27, len(df_raw)
display(df_raw.sort_values(["model", "dataset", "seed"])[
    ["_file", "model", "dataset", "seed", "test_accuracy", "T_star_hard", "T_star_soft", "gap"]
])


raw/: 27 files; raw_v2/: 27 files


,_file,model,dataset,seed,test_accuracy,T_star_hard,T_star_soft,gap
1,results_bert-base-uncased_chaosnli_m_seed42.json,bert-base-uncased,ChaosNLI-M,42,0.540000,1.228498,4.216779,0.114137
0,results_bert-base-uncased_chaosnli_m_seed123.json,bert-base-uncased,ChaosNLI-M,123,0.521250,1.198224,4.137025,0.125476
2,results_bert-base-uncased_chaosnli_m_seed456.json,bert-base-uncased,ChaosNLI-M,456,0.531250,1.241589,4.039873,0.118419
4,results_bert-base-uncased_chaosnli_s_seed42.json,bert-base-uncased,ChaosNLI-S,42,0.729194,1.267239,2.853706,0.047087
3,results_bert-base-uncased_chaosnli_s_seed123.json,bert-base-uncased,ChaosNLI-S,123,0.704095,1.333897,3.098190,0.049358
5,results_bert-base-uncased_chaosnli_s_seed456.json,bert-base-uncased,ChaosNLI-S,456,0.704095,1.420271,3.429240,0.052711
7,results_bert-large-uncased_chaosnli_m_seed42.json,bert-large-uncased,ChaosNLI-M,42,0.535000,1.196403,3.324129,0.069024
6,results_bert-large-uncased_chaosnli_m_seed123....,bert-large-uncased,ChaosNLI-M,123,0.515000,1.036246,2.994772,0.079128
8,results_bert-large-uncased_chaosnli_m_seed456....,bert-large-uncased,ChaosNLI-M,456,0.517500,1.135218,3.089568,0.074013
10,results_bert-large-uncased_chaosnli_s_seed42.json,bert-large-uncased,ChaosNLI-S,42,0.558785,0.891674,2.105762,0.056977


## 2. Validation: silent optimizer failures

Flag any run where `T_star_hard` or `T_star_soft` equals the L-BFGS starting value `1.0` exactly, or where `gap` equals `0.0` exactly.


In [3]:
OPT_X0 = 1.0

def validate_runs(df: pd.DataFrame, label: str) -> pd.DataFrame:
    flags = []
    for _, r in df.iterrows():
        problems = []
        if float(r["T_star_hard"]) == OPT_X0:
            problems.append("T_star_hard==x0")
        if float(r["T_star_soft"]) == OPT_X0:
            problems.append("T_star_soft==x0")
        if float(r["gap"]) == 0.0:
            problems.append("gap==0.0")
        if problems:
            flags.append({
                "source": label,
                "file": r["_file"],
                "model": r["model"],
                "dataset": r["dataset"],
                "seed": int(r["seed"]),
                "T_star_hard": r["T_star_hard"],
                "T_star_soft": r["T_star_soft"],
                "gap": r["gap"],
                "problems": ", ".join(problems),
            })
    return pd.DataFrame(flags)

bad_raw = validate_runs(df_raw, "raw")
bad_v2 = validate_runs(df_v2, "raw_v2")
frames = [x for x in (bad_raw, bad_v2) if len(x)]
bad = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(bad):
    display(bad)
    raise AssertionError(f"Silent optimizer failure signature detected in {len(bad)} run(s)")
print("PASS: no T==1.0 or gap==0.0 exact failures in raw/ or raw_v2/")


PASS: no T==1.0 or gap==0.0 exact failures in raw/ or raw_v2/


## 3. Independent aggregation (cross-check of `aggregate.py`)


In [4]:
METRICS = [
    "test_accuracy", "uncal_ece", "ts_hard_ece", "ts_soft_ece",
    "uncal_bs_soft", "ts_hard_bs_soft", "ts_soft_bs_soft",
    "gap", "T_star_hard", "T_star_soft",
    "iso_hard_ece", "iso_soft_ece", "iso_hard_bs_soft", "iso_soft_bs_soft", "iso_gap",
]

def aggregate_independent(df: pd.DataFrame) -> dict:
    grouped = defaultdict(list)
    for _, r in df.iterrows():
        dataset = r["dataset"]
        if dataset is None or (isinstance(dataset, float) and np.isnan(dataset)):
            dataset = "CIFAR-10H" if "resnet" in str(r["model"]) else "UNKNOWN"
        grouped[(r["model"], dataset)].append(r.to_dict())
    out = {}
    for (model, dataset), runs in sorted(grouped.items()):
        row = {"model": model, "dataset": dataset, "n_runs": len(runs)}
        for m in METRICS:
            vals = []
            for run in runs:
                if m in run and run[m] is not None:
                    try:
                        v = float(run[m])
                    except (TypeError, ValueError):
                        continue
                    if np.isfinite(v):
                        vals.append(v)
            if vals:
                row[f"{m}_mean"] = float(np.mean(vals))
                row[f"{m}_std"] = float(np.std(vals))
        out[f"{model}_{dataset}"] = row
    return out

agg_raw = aggregate_independent(df_raw)
agg_v2 = aggregate_independent(df_v2)

for prefix, ours in [("final_results", agg_raw), ("final_results_v2", agg_v2)]:
    path = REPO / "results" / "tables" / f"{prefix}.json"
    theirs = json.loads(path.read_text())
    mismatches = []
    for key, row in ours.items():
        if key not in theirs:
            mismatches.append(f"missing key {key}")
            continue
        for m in ["test_accuracy", "gap", "T_star_hard", "T_star_soft",
                  "ts_hard_bs_soft", "ts_soft_bs_soft"]:
            for suf in ["_mean", "_std"]:
                a, b = row.get(m + suf), theirs[key].get(m + suf)
                if a is None or b is None:
                    continue
                if abs(a - b) > 1e-12:
                    mismatches.append(f"{key}.{m}{suf}: {a} vs {b}")
    assert not mismatches, mismatches
    print(f"PASS: independent agg matches results/tables/{prefix}.json on TS fields")

for m in ["test_accuracy", "gap", "T_star_hard", "T_star_soft", "ts_hard_bs_soft", "ts_soft_bs_soft"]:
    a = agg_raw["resnet18_CIFAR-10H"][m + "_mean"]
    b = agg_v2["resnet18_CIFAR-10H"][m + "_mean"]
    assert abs(a - b) < 1e-12, (m, a, b)
print("PASS: raw and raw_v2 agree on ResNet-18 TS aggregates")


PASS: independent agg matches results/tables/final_results.json on TS fields
PASS: independent agg matches results/tables/final_results_v2.json on TS fields
PASS: raw and raw_v2 agree on ResNet-18 TS aggregates


## 4. Reproduce Table 1 (vision) and diff against `main.tex`

Table 1 is taken from `results/raw/` aggregates (identical to `raw_v2` for vision after the ResNet-18 seed-42 fix).


In [5]:
from decimal import Decimal, ROUND_HALF_UP

def round3(x: float) -> float:
    """Publication rounding: half-up to 3 decimals (not banker's round)."""
    return float(Decimal(str(x)).quantize(Decimal("0.001"), rounding=ROUND_HALF_UP))

def fmt3(mean, std):
    return f"{round3(mean):.3f} ({round3(std):.3f})"

def parse_vision_table(tex: str):
    m = re.search(r"\\label\{tab:vision\}.*?\\midrule(.*?)\\bottomrule", tex, re.S)
    assert m, "tab:vision not found"
    body = m.group(1)
    rows = {}
    for model in ["ResNet-18", "ResNet-50", "ResNet-101"]:
        rm = re.search(
            rf"{re.escape(model)}\s*((?:&\s*[0-9.]+\s*\([0-9.]+\)\s*){{7}})\\\\",
            body,
        )
        assert rm, f"row {model} not found in Table 1"
        vals = re.findall(r"([0-9.]+)\s*\(([0-9.]+)\)", rm.group(1))
        assert len(vals) == 7, vals
        keys = ["Acc", "Uncal ECE", "TS-H ECE", "TS-S ECE", "TS-H BS", "TS-S BS", "Gap"]
        rows[model] = {k: (float(a), float(b)) for k, (a, b) in zip(keys, vals)}
    return rows

tex = TEX.read_text()
paper_t1 = parse_vision_table(tex)

model_map = {
    "ResNet-18": "resnet18_CIFAR-10H",
    "ResNet-50": "resnet50_CIFAR-10H",
    "ResNet-101": "resnet101_CIFAR-10H",
}
metric_map = [
    ("Acc", "test_accuracy"),
    ("Uncal ECE", "uncal_ece"),
    ("TS-H ECE", "ts_hard_ece"),
    ("TS-S ECE", "ts_soft_ece"),
    ("TS-H BS", "ts_hard_bs_soft"),
    ("TS-S BS", "ts_soft_bs_soft"),
    ("Gap", "gap"),
]

t1_diffs = []
print("Table 1 (computed from raw/ vs paper/main.tex):")
for model, key in model_map.items():
    row = agg_raw[key]
    for pname, mname in metric_map:
        computed = (round3(row[mname + "_mean"]), round3(row[mname + "_std"]))
        paper = paper_t1[model][pname]
        ok = computed == paper
        status = "OK" if ok else "DIFF"
        print(f"  {status} {model} {pname}: data={fmt3(*computed)}  paper={fmt3(*paper)}")
        if not ok:
            t1_diffs.append((model, pname, computed, paper))

assert not t1_diffs, t1_diffs
print("PASS: Table 1 matches main.tex with zero discrepancies")


Table 1 (computed from raw/ vs paper/main.tex):
  OK ResNet-18 Acc: data=0.925 (0.002)  paper=0.925 (0.002)
  OK ResNet-18 Uncal ECE: data=0.034 (0.001)  paper=0.034 (0.001)
  OK ResNet-18 TS-H ECE: data=0.013 (0.001)  paper=0.013 (0.001)
  OK ResNet-18 TS-S ECE: data=0.022 (0.002)  paper=0.022 (0.002)
  OK ResNet-18 TS-H BS: data=0.093 (0.000)  paper=0.093 (0.000)
  OK ResNet-18 TS-S BS: data=0.091 (0.001)  paper=0.091 (0.001)
  OK ResNet-18 Gap: data=0.002 (0.000)  paper=0.002 (0.000)
  OK ResNet-50 Acc: data=0.886 (0.003)  paper=0.886 (0.003)
  OK ResNet-50 Uncal ECE: data=0.036 (0.003)  paper=0.036 (0.003)
  OK ResNet-50 TS-H ECE: data=0.020 (0.002)  paper=0.020 (0.002)
  OK ResNet-50 TS-S ECE: data=0.022 (0.004)  paper=0.022 (0.004)
  OK ResNet-50 TS-H BS: data=0.139 (0.003)  paper=0.139 (0.003)
  OK ResNet-50 TS-S BS: data=0.136 (0.004)  paper=0.136 (0.004)
  OK ResNet-50 Gap: data=0.003 (0.001)  paper=0.003 (0.001)
  OK ResNet-101 Acc: data=0.908 (0.008)  paper=0.908 (0.008)
  O

## 5. Reproduce Table 2 (language) and diff against `main.tex`

Language Table 2 (including independent BERT-large SNLI) is verified from `results/raw_v2/`.


In [6]:
def parse_language_table(tex: str):
    m = re.search(r"\\label\{tab:language\}.*?\\midrule(.*?)\\bottomrule", tex, re.S)
    assert m, "tab:language not found"
    body = m.group(1)
    rows = {}
    for model, short in [("DistilBERT", "distilbert-base-uncased"),
                         ("BERT-base", "bert-base-uncased"),
                         ("BERT-large", "bert-large-uncased")]:
        for ds, dscode in [("S", "ChaosNLI-S"), ("M", "ChaosNLI-M")]:
            rm = re.search(
                rf"{re.escape(model)}\s*&\s*{ds}\s*((?:&\s*[0-9.]+\s*\([0-9.]+\)\s*){{6}})\\\\",
                body,
            )
            assert rm, f"row {model} {ds} not found"
            vals = re.findall(r"([0-9.]+)\s*\(([0-9.]+)\)", rm.group(1))
            assert len(vals) == 6, vals
            keys = ["Acc", "TS-H ECE", "TS-S ECE", "TS-H BS", "TS-S BS", "Gap"]
            rows[(short, dscode)] = {k: (float(a), float(b)) for k, (a, b) in zip(keys, vals)}
    return rows

paper_t2 = parse_language_table(tex)
lang_metric_map = [
    ("Acc", "test_accuracy"),
    ("TS-H ECE", "ts_hard_ece"),
    ("TS-S ECE", "ts_soft_ece"),
    ("TS-H BS", "ts_hard_bs_soft"),
    ("TS-S BS", "ts_soft_bs_soft"),
    ("Gap", "gap"),
]

t2_diffs = []
print("Table 2 (computed from raw_v2/ vs paper/main.tex):")
for (model, dataset), paper_row in paper_t2.items():
    key = f"{model}_{dataset}"
    row = agg_v2[key]
    for pname, mname in lang_metric_map:
        computed = (round3(row[mname + "_mean"]), round3(row[mname + "_std"]))
        paper = paper_row[pname]
        ok = computed == paper
        status = "OK" if ok else "DIFF"
        print(f"  {status} {model} {dataset} {pname}: data={fmt3(*computed)}  paper={fmt3(*paper)}")
        if not ok:
            t2_diffs.append((model, dataset, pname, computed, paper))

assert not t2_diffs, t2_diffs
print("PASS: Table 2 matches main.tex with zero discrepancies")


Table 2 (computed from raw_v2/ vs paper/main.tex):
  OK distilbert-base-uncased ChaosNLI-S Acc: data=0.748 (0.010)  paper=0.748 (0.010)
  OK distilbert-base-uncased ChaosNLI-S TS-H ECE: data=0.091 (0.010)  paper=0.091 (0.010)
  OK distilbert-base-uncased ChaosNLI-S TS-S ECE: data=0.130 (0.008)  paper=0.130 (0.008)
  OK distilbert-base-uncased ChaosNLI-S TS-H BS: data=0.184 (0.005)  paper=0.184 (0.005)
  OK distilbert-base-uncased ChaosNLI-S TS-S BS: data=0.140 (0.004)  paper=0.140 (0.004)
  OK distilbert-base-uncased ChaosNLI-S Gap: data=0.045 (0.001)  paper=0.045 (0.001)
  OK distilbert-base-uncased ChaosNLI-M Acc: data=0.501 (0.014)  paper=0.501 (0.014)
  OK distilbert-base-uncased ChaosNLI-M TS-H ECE: data=0.241 (0.015)  paper=0.241 (0.015)
  OK distilbert-base-uncased ChaosNLI-M TS-S ECE: data=0.041 (0.005)  paper=0.041 (0.005)
  OK distilbert-base-uncased ChaosNLI-M TS-H BS: data=0.302 (0.011)  paper=0.302 (0.011)
  OK distilbert-base-uncased ChaosNLI-M TS-S BS: data=0.169 (0.004)

## 6. Reproduce Table 3 (isotonic) and diff against `main.tex`


In [7]:
def parse_iso_table(tex: str):
    m = re.search(r"\\label\{tab:iso\}.*?\\midrule(.*?)\\bottomrule", tex, re.S)
    assert m, "tab:iso not found"
    body = m.group(1)
    patterns = [
        ("DistilBERT", "ChaosNLI-S", "distilbert-base-uncased_ChaosNLI-S"),
        ("BERT-base", "ChaosNLI-S", "bert-base-uncased_ChaosNLI-S"),
        ("BERT-large", "ChaosNLI-S", "bert-large-uncased_ChaosNLI-S"),
        ("DistilBERT", "ChaosNLI-M", "distilbert-base-uncased_ChaosNLI-M"),
        ("BERT-base", "ChaosNLI-M", "bert-base-uncased_ChaosNLI-M"),
        ("BERT-large", "ChaosNLI-M", "bert-large-uncased_ChaosNLI-M"),
        ("ResNet-18", "CIFAR-10H", "resnet18_CIFAR-10H"),
        ("ResNet-50", "CIFAR-10H", "resnet50_CIFAR-10H"),
        ("ResNet-101", "CIFAR-10H", "resnet101_CIFAR-10H"),
    ]
    rows = {}
    for model, dataset, key in patterns:
        rm = re.search(
            rf"{re.escape(model)}\s*&\s*{re.escape(dataset)}\s*&\s*([0-9.]+)\s*&\s*([0-9.]+)\s*\\\\",
            body,
        )
        assert rm, f"iso row {model} {dataset} not found"
        rows[key] = (float(rm.group(1)), float(rm.group(2)))
    return rows

paper_t3 = parse_iso_table(tex)
t3_diffs = []
print("Table 3 (TS gap / ISO gap from raw_v2/ vs paper):")
for key, (p_ts, p_iso) in paper_t3.items():
    row = agg_v2[key]
    c_ts = round3(row["gap_mean"])
    c_iso = round3(row["iso_gap_mean"])
    ok = (c_ts, c_iso) == (p_ts, p_iso)
    status = "OK" if ok else "DIFF"
    print(f"  {status} {key}: data=({c_ts}, {c_iso}) paper=({p_ts}, {p_iso})")
    if not ok:
        t3_diffs.append((key, (c_ts, c_iso), (p_ts, p_iso)))

assert not t3_diffs, t3_diffs
r18 = agg_v2["resnet18_CIFAR-10H"]
print(f"ResNet-18 iso_gap={r18['iso_gap_mean']:.6f} vs ts_gap={r18['gap_mean']:.6f} (both ~0.002 at 3dp)")
print("PASS: Table 3 matches main.tex with zero discrepancies")


Table 3 (TS gap / ISO gap from raw_v2/ vs paper):
  OK distilbert-base-uncased_ChaosNLI-S: data=(0.045, 0.054) paper=(0.045, 0.054)
  OK bert-base-uncased_ChaosNLI-S: data=(0.05, 0.055) paper=(0.05, 0.055)
  OK bert-large-uncased_ChaosNLI-S: data=(0.053, 0.069) paper=(0.053, 0.069)
  OK distilbert-base-uncased_ChaosNLI-M: data=(0.134, 0.136) paper=(0.134, 0.136)
  OK bert-base-uncased_ChaosNLI-M: data=(0.119, 0.126) paper=(0.119, 0.126)
  OK bert-large-uncased_ChaosNLI-M: data=(0.074, 0.101) paper=(0.074, 0.101)
  OK resnet18_CIFAR-10H: data=(0.002, 0.002) paper=(0.002, 0.002)
  OK resnet50_CIFAR-10H: data=(0.003, 0.002) paper=(0.003, 0.002)
  OK resnet101_CIFAR-10H: data=(0.003, 0.003) paper=(0.003, 0.003)
ResNet-18 iso_gap=0.002244 vs ts_gap=0.002347 (both ~0.002 at 3dp)
PASS: Table 3 matches main.tex with zero discrepancies


## 7. Derived prose claims (Sections 5.1, 5.4, 5.6)

Each claim is printed next to the exact sentence from `main.tex` it verifies.


In [8]:
vision_keys = [k for k in agg_v2 if "CIFAR" in k]
lang_keys = [k for k in agg_v2 if "Chaos" in k]
vis_mean = float(np.mean([agg_v2[k]["gap_mean"] for k in vision_keys]))
lang_mean = float(np.mean([agg_v2[k]["gap_mean"] for k in lang_keys]))
ratio_nx = lang_mean / vis_mean

ratios = {k: agg_v2[k]["T_star_soft_mean"] / agg_v2[k]["T_star_hard_mean"] for k in agg_v2}
vis_ratios = {k: ratios[k] for k in vision_keys}
min_r_key = min(ratios, key=ratios.get)
max_r_key = max(ratios, key=ratios.get)
min_r, max_r = ratios[min_r_key], ratios[max_r_key]
vis_min_r, vis_max_r = min(vis_ratios.values()), max(vis_ratios.values())

eligible = exceed8 = exceed18 = 0
for k, row in agg_v2.items():
    g, s = row["gap_mean"], row["gap_std"]
    if round(s, 3) == 0.0:
        continue
    eligible += 1
    r = g / s
    if r > 8:
        exceed8 += 1
    if r > 18:
        exceed18 += 1

def find_snippet(substr: str):
    """Return a cleaned single-line snippet containing substr, or None."""
    idx = tex.find(substr)
    if idx < 0:
        return None
    # expand to surrounding sentence-ish window
    start = tex.rfind("\n", 0, idx)
    start = 0 if start < 0 else start + 1
    end = tex.find("\n", idx)
    end = len(tex) if end < 0 else end
    return re.sub(r"\s+", " ", tex[start:end]).strip()

claims = []

sent = find_snippet("mean gap 0.003")
claims.append({
    "claim": "vision mean gap rounds to 0.003 at 3 decimals",
    "computed": f"{vis_mean:.6f} -> {round(vis_mean, 3):.3f}",
    "paper_sentence": sent if sent and "0.079" in sent else find_snippet("language domain (mean gap 0.079)"),
    "ok": round(vis_mean, 3) == 0.003 and "mean gap 0.003" in tex,
})

sent = find_snippet("ratio ranges from 1.26 to 1.32")
claims.append({
    "claim": "Sec 5.1 vision T_oracle/T_hard range",
    "computed": f"{vis_min_r:.2f} to {vis_max_r:.2f}",
    "paper_sentence": sent,
    "ok": abs(vis_min_r - 1.26) < 0.005 and abs(vis_max_r - 1.32) < 0.005 and sent is not None,
})

sent = find_snippet("from 1.26 (ResNet-18) to 3.48")
claims.append({
    "claim": "Sec 5.4 full T_oracle/T_hard range",
    "computed": f"{min_r:.2f} ({min_r_key}) to {max_r:.2f} ({max_r_key})",
    "paper_sentence": sent,
    "ok": sent is not None and abs(min_r - 1.26) < 0.005 and abs(max_r - 3.48) < 0.005,
})

sent = find_snippet("five of the seven configurations")
claims.append({
    "claim": "Sec 5.6 gap/std exceeds-8 / exceeds-18",
    "computed": f"eligible={eligible}, exceed8={exceed8}, exceed18={exceed18}",
    "paper_sentence": sent,
    "ok": eligible == 7 and exceed8 == 5 and exceed18 == 4 and sent is not None and "four language cases" in tex,
})

sent = find_snippet("about 28 times the mean vision gap")
claims.append({
    "claim": "~28x language/vision mean gap ratio",
    "computed": f"{ratio_nx:.2f}x (lang={lang_mean:.6f}, vis={vis_mean:.6f})",
    "paper_sentence": sent,
    "ok": abs(ratio_nx - 28) < 1.0 and sent is not None,
})

for c in claims:
    status = "OK" if c["ok"] else "DIFF"
    print(f"\n[{status}] {c['claim']}")
    print(f"  computed: {c['computed']}")
    print(f"  paper:    {c['paper_sentence']}")

assert all(c["ok"] for c in claims), [c for c in claims if not c["ok"]]
print("\nPASS: all derived prose claims match main.tex")



[OK] vision mean gap rounds to 0.003 at 3 decimals
  computed: 0.002831 -> 0.003
  paper:    language domain (mean gap 0.079) than in vision (mean gap 0.003). A

[OK] Sec 5.1 vision T_oracle/T_hard range
  computed: 1.26 to 1.32
  paper:    $T_{\mathrm{oracle}}/T_{\mathrm{hard}}$ ratio ranges from 1.26 to 1.32,

[OK] Sec 5.4 full T_oracle/T_hard range
  computed: 1.26 (resnet18_CIFAR-10H) to 3.48 (distilbert-base-uncased_ChaosNLI-M)
  paper:    configurations, from 1.26 (ResNet-18) to 3.48 (DistilBERT on

[OK] Sec 5.6 gap/std exceeds-8 / exceeds-18
  computed: eligible=7, exceed8=5, exceed18=4
  paper:    temperature scaling exceed 8 in five of the seven configurations whose

[OK] ~28x language/vision mean gap ratio
  computed: 27.96x (lang=0.079151, vis=0.002831)
  paper:    language gap (0.079) is about 28 times the mean vision gap (0.003) when

PASS: all derived prose claims match main.tex


## 8. Acceptance summary

If every cell above printed `PASS`, the released JSON, regenerated aggregates, and `paper/main.tex` are consistent.


In [9]:
print("=" * 60)
print("ALL CHECKS PASSED")
print("  - 27 raw files loaded; no silent T=1.0 / gap=0.0 failures")
print("  - Independent aggregation matches results/tables/")
print("  - Tables 1, 2, 3 match paper/main.tex exactly")
print("  - Derived Section 5.1 / 5.4 / 5.6 claims match")
print("=" * 60)


ALL CHECKS PASSED
  - 27 raw files loaded; no silent T=1.0 / gap=0.0 failures
  - Independent aggregation matches results/tables/
  - Tables 1, 2, 3 match paper/main.tex exactly
  - Derived Section 5.1 / 5.4 / 5.6 claims match
